In [ ]:
# Régi proto függvény

def get_all_stays(search_city):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import time
    import re

    # ----------------- 1️⃣ Selenium setup -----------------
    chrome_options = Options()
    #chrome_options.add_argument("--headless")  # Fej nélküli mód, ha kell
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 15)

    # ----------------- 2️⃣ Megnyitjuk a főoldalt -----------------
    driver.get("https://www.cozycozy.com/en")

    # ----------------- 3️⃣ Cookie elfogadás -----------------
    try:
        accept_btn = wait.until(EC.element_to_be_clickable((By.ID, "accept-btn")))
        accept_btn.click()
        print("Cookie elfogadva")
    except:
        print("Nincs cookie gomb")

    # ----------------- 4️⃣ Hely kitöltése és autocomplete választás -----------------
    location_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[formcontrolname='location']")))
    location_input.clear()
    location_input.send_keys(search_city)

    # Várjuk az autocomplete felugró div-et
    autocomplete_first = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.autocomplete .place-prediction.selected")))
    autocomplete_text = autocomplete_first.text
    autocomplete_first.click()
    print("Kiválasztott hely:", autocomplete_text)

    # ----------------- 5️⃣ Dátumok választása -----------------

    def select_date(day, month, year):
        # Várjuk, hogy az overlay panel megjelenjen
        overlay = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "cdk-overlay-pane"))
        )

        # Keresés a megfelelő nap gombjára az overlay-en belül
        day_btn = WebDriverWait(overlay, 10).until(
            EC.element_to_be_clickable((
                By.XPATH,
                f".//button[contains(@class,'pika-day') and @data-pika-day='{day}' and @data-pika-month='{month-1}' and @data-pika-year='{year}']"
            ))
        )
        
        # JS click a biztos kattintásért
        driver.execute_script("arguments[0].click();", day_btn)


    # Érkezés
    arrival_input = driver.find_elements(By.CSS_SELECTOR, "label.date input")[0]
    driver.execute_script("arguments[0].click();", arrival_input)
    select_date(21, 12, 2025)

    # Távozás
    departure_input = driver.find_elements(By.CSS_SELECTOR, "label.date input")[1]
    driver.execute_script("arguments[0].click();", departure_input)
    select_date(24, 12, 2025)



    # ----------------- 6️⃣ Következő gomb a naptárnál -----------------
    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.next")))
        next_btn.click()
    except:
        pass

    # ----------------- 7️⃣ Szobák/felnőttek overlay OK -----------------
    try:
        print("🔹 Várakozás az overlay-re...")
        overlay = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "cdk-overlay-pane"))
        )

        ok_btn = WebDriverWait(overlay, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.a-btn"))
        )
        
        ok_btn.click()
    except Exception as e:
        print("❌ Hiba történt az OK gombnál:", e)

    # ----------------- UNCHECK 'Összehasonlítás: cozycozy és booking.com' -----------------
    try:
        print("🔹 Ellenőrizzük az összehasonlító checkbox állapotát...")
        
        # Keressük a checkbox input elemet
        compare_checkbox = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section.compare-sites input[type='checkbox']"))
        )

        # Ha be van pipálva, kattintsunk rá, hogy UNCHECK legyen
        if compare_checkbox.is_selected():
            print("🔹 Checkbox be van pipálva. UNCHECKeljük...")
            driver.execute_script("arguments[0].click();", compare_checkbox)
            print("✅ Checkbox UNCHECKelve.")
        else:
            print("🔹 Checkbox már UNCHECKelve.")

    except Exception as e:
        print("❌ Hiba a compare-sites checkbox kezelésekor:", e)


    # ----------------- 8️⃣ Keresés gomb -----------------
    search_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.form-button")))
    search_btn.click()

    time.sleep(7)

    # Debug: aktuális URL
    current_url = driver.current_url
    print("🔹 Jelenlegi oldal URL:", current_url)


    # ----------------- 9️⃣ Várakozás a szálláslista megjelenésére -----------------
    try:
        print("🔹 Várakozás legalább egy 'Megnézem' linkre (a.m-card-button)...")
        links = WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.m-card-button"))
        )
        print(f"✅ Talált {len(links)} 'Megnézem' linket.")
    except Exception as e:
        print("❌ Hiba: szálláslista nem töltődött be időben:", e)

    # ----------------- 9️⃣b Filter gomb ellenőrzés -----------------
    try:
        print("🔹 Várakozás a Filter gombra...")
        filter_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div.button.filter.badge button"))
        )
        print("✅ Filter gomb kattintható.")
        print("🔹 Filter gomb szöveg:", filter_btn.text)
    except Exception as e:
        print("❌ Hiba a Filter gombnál:", e)

    # ----------------- 🔟 Kinyerjük a searchId-t -----------------
    try:
        found = False
        for link in driver.find_elements(By.CSS_SELECTOR, "a.m-card-button"):
            href = link.get_attribute("href")
            if "searchId=" in href:
                match = re.search(r"searchId=([A-Za-z0-9]+)", href)
                if match:
                    search_id = match.group(1)
                    print("✅ Talált searchId:", search_id)
                    
                    import json

                    # API hívás közvetlenül a böngészőből JavaScript-tel
                    api_call_script = f"""
                    return fetch('https://www.cozycozy.com/api/getResultList', {{
                        method: 'POST',
                        headers: {{
                            'accept': 'application/json, text/plain, */*',
                            'content-type': 'application/json',
                            'x-search-id': '{search_id}',
                            'x-split-id': '0'
                        }},
                        body: JSON.stringify({{
                            searchId: '{search_id}',
                            sorting: 'ranking',
                            offset: 0,
                            count: 50,
                            filters: {{
                                bounds: null,
                                noBounds: false,
                                price: [-0.5, 9007199254740991],
                                instantBooking: true,
                                combinedTypeCodes: [],
                                starRatings: [],
                                minRating: 0,
                                ratingRequired: false,
                                amenityCodes: [],
                                providerCodes: [],
                                minBedRoomCount: 1,
                                minBathRoomCount: 0,
                                cityCodes: [],
                                areaCodes: [],
                                minResponseTime: null,
                                updateBounds: true,
                                breakfast: false,
                                minCancellationCategory: 0
                            }},
                            estimateBounds: {{
                                targetSize: {{
                                    width: 1136,
                                    height: 925
                                }}
                            }},
                            prefixAccommodationIds: [],
                            processNewResults: true,
                            columnCount: 3,
                            excludeAds: false
                        }})
                    }}).then(response => response.json());
                    """

                    print("🔹 API hívás indítása...")
                    results = driver.execute_script(api_call_script)
                    print("✅ Eredmények:", json.dumps(results, indent=2))

                    found = True
                    break


        if not found:
            print("❌ Nem található searchId a linkekben.")
            driver.quit()
            exit()

    except Exception as e:
        print("❌ Hiba a searchId keresésekor:", e)
        driver.quit()

        return {}

    driver.quit()

    return results

#results = get_all_stays("Alicante")

In [1]:
# Új, gyorsabb (nincs manuális keresés)

def get_all_stays(city, country, start_date, end_date, 
                  rooms=1, adults=2, children=0,
                  price_min=0, price_max=9007199254740991,
                  min_rating=0,
                  accommodation_types=None,
                  amenities=None,
                  breakfast=False):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import time
    import re

    # Type kódok mapping
    type_mapping = {
        'hotel': 'hotel',
        'vr': 'vr',  # vacation rental
        'hostel': 'hostel',
        'guest': 'guest',  # guest house
        'ooo': 'ooo',  # egyéb
        'camping': 'camping'
    }

    # Type kódok - NAGYBETŰVEL és $ előtaggal!
    if accommodation_types is None:
        combined_types = ["$HOTEL", "$VR", "$HOSTEL", "$GUEST", "$OOO", "$CAMPING"]
    else:
        combined_types = [f"${t.upper()}" for t in accommodation_types]

    # Amenity kódok - NAGYBETŰVEL!
    if amenities is None:
        amenity_codes = []
    else:
        amenity_codes = [a.upper() for a in amenities]

    def build_filter_string(price_min, price_max, min_rating, accommodation_types, amenities, breakfast):
        filters = []
        
        # Price range
        if price_min > 0 or price_max < 9007199254740991:
            filters.append(f"p:{price_min},{price_max}")
        
        # Rating
        if min_rating > 0:
            filters.append(f"r:{min_rating}")
        
        # Types
        if accommodation_types:
            type_str = ','.join([f"${t}" for t in accommodation_types])
            filters.append(f"t:{type_str}")
        
        # Amenities
        if amenities:
            amenity_str = ','.join(amenities)
            filters.append(f"a:{amenity_str}")
        
        # Breakfast
        if breakfast:
            filters.append("b:1")
        
        return ';'.join(filters)

    filter_string = build_filter_string(price_min, price_max, min_rating, 
                                        accommodation_types, amenities, breakfast)

    # ----------------- 1️⃣ Selenium setup -----------------
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Fej nélküli mód, ha kell
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 15)

    # ----------------- 2️⃣ Megnyitjuk a főoldalt -----------------
    base_url = f"https://www.cozycozy.com/en/search/{city}%2C%20{country}/{start_date}/{end_date}/{rooms}-{adults}-{children}/results"

    if filter_string:
        url = f"{base_url}?filters={filter_string}"
    else:
        url = base_url

    driver.get(url)
    
    # Debug: aktuális URL
    current_url = driver.current_url
    print("🔹 Jelenlegi oldal URL:", current_url)


    # ----------------- 9️⃣ Várakozás a szálláslista megjelenésére -----------------
    try:
        print("🔹 Várakozás legalább egy 'Megnézem' linkre (a.m-card-button)...")
        links = WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.m-card-button"))
        )
        print(f"✅ Talált {len(links)} 'Megnézem' linket.")
    except Exception as e:
        print("❌ Hiba: szálláslista nem töltődött be időben:", e)

    # ----------------- 9️⃣b Filter gomb ellenőrzés -----------------
    try:
        print("🔹 Várakozás a Filter gombra...")
        filter_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div.button.filter.badge button"))
        )
        print("✅ Filter gomb kattintható.")
        print("🔹 Filter gomb szöveg:", filter_btn.text)
    except Exception as e:
        print("❌ Hiba a Filter gombnál:", e)

    # ----------------- 🔟 Kinyerjük a searchId-t -----------------
    try:
        found = False
        for link in driver.find_elements(By.CSS_SELECTOR, "a.m-card-button"):
            href = link.get_attribute("href")
            if "searchId=" in href:
                match = re.search(r"searchId=([A-Za-z0-9]+)", href)
                if match:
                    search_id = match.group(1)
                    print("✅ Talált searchId:", search_id)
                    
                    import json

                    # API hívás közvetlenül a böngészőből JavaScript-tel
                    api_call_script = f"""
                    return fetch('https://www.cozycozy.com/api/getResultList', {{
                        method: 'POST',
                        headers: {{
                            'accept': 'application/json, text/plain, */*',
                            'content-type': 'application/json',
                            'x-search-id': '{search_id}',
                            'x-split-id': '0'
                        }},
                        body: JSON.stringify({{
                            searchId: '{search_id}',
                            sorting: 'ranking',
                            offset: 0,
                            count: 90,
                            filters: {{
                                bounds: null,
                                noBounds: true,
                                price: [{price_min}, {price_max}],
                                instantBooking: true,
                                combinedTypeCodes: {json.dumps(combined_types)},
                                starRatings: [],
                                minRating: {min_rating},
                                ratingRequired: {str(min_rating > 0).lower()},
                                amenityCodes: {json.dumps(amenity_codes)},
                                providerCodes: [],
                                minBedRoomCount: 1,
                                minBathRoomCount: 0,
                                cityCodes: [],
                                areaCodes: [],
                                minResponseTime: null,
                                updateBounds: true,
                                breakfast: {str(breakfast).lower()},
                                minCancellationCategory: 0
                            }},
                            estimateBounds: {{
                                targetSize: {{
                                    width: 1136,
                                    height: 925
                                }}
                            }},
                            prefixAccommodationIds: [],
                            processNewResults: true,
                            columnCount: 3,
                            excludeAds: false
                        }})
                    }}).then(response => response.json());
                    """

                    print("🔹 API hívás indítása...")
                    results = driver.execute_script(api_call_script)
                    print("✅ Eredmények:", len(results['entries']))

                    found = True
                    break


        if not found:
            print("❌ Nem található searchId a linkekben.")
            driver.quit()
            exit()

    except Exception as e:
        print("❌ Hiba a searchId keresésekor:", e)
        driver.quit()

        return {}

    driver.quit()

    return results

results = get_all_stays(
    "Budapest", "Hungary",
    "2026-06-15", "2026-06-20",
    price_max=5*25000/385,
    #accommodation_types=['hotel'],
    #amenities=['swimpool'],
    min_rating=70
)

🔹 Jelenlegi oldal URL: https://www.cozycozy.com/en/search/Budapest%2C%20Hungary/2026-06-15/2026-06-20/1-2-0/progress?then=%2Fen%2Fsearch%2FBudapest%252C%2520Hungary%2F2026-06-15%2F2026-06-20%2F1-2-0%2Fresults%3Ffilters%3Dp:0,324.6753246753247;r:70
🔹 Várakozás legalább egy 'Megnézem' linkre (a.m-card-button)...
✅ Talált 10 'Megnézem' linket.
🔹 Várakozás a Filter gombra...
✅ Filter gomb kattintható.
🔹 Filter gomb szöveg: 3
Filters
✅ Talált searchId: bQ6UNmH905eZkmHu
🔹 API hívás indítása...
✅ Eredmények: 90


In [2]:
# Dataframe-be rendezés

import pandas as pd

def parse_cozycozy_results(results):
    """
    Cozycozy API eredmények DataFrame-be konvertálása.
    
    Args:
        results (dict): A Cozycozy API JSON válasza
        
    Returns:
        pd.DataFrame: Szállások adatai táblázatban
    """
    
    if not results or 'entries' not in results:
        print("❌ Nincs 'entries' kulcs az eredményekben!")
        return pd.DataFrame()
    
    parsed_data = []
    
    for entry in results['entries']:
        # Alapadatok
        base_info = {
            'accommodation_id': entry.get('accommodationId'),
            'name': entry.get('name'),
            'title': entry.get('title'),
            'subtitle': entry.get('subTitle'),
            'city': entry.get('cityName'),
            'rating_score': entry.get('ratingScore'),
            'rating_count': entry.get('ratingCount'),
            'location_text': entry.get('locationText'),
            'surface_text': entry.get('surfaceText'),
            'latitude': entry.get('coordinates', {}).get('latitude'),
            'longitude': entry.get('coordinates', {}).get('longitude'),
            'cancellation_category': entry.get('cancellationCategory'),
            'instant_booking': entry.get('instantBooking'),
            'result_count': entry.get('resultCount')
        }
        
        # A legolcsóbb ajánlat kiválasztása
        highlighted = entry.get('highlightedResults', [])
        if highlighted:
            cheapest = min(highlighted, key=lambda x: x.get('eurPricePerNight', float('inf')))
            
            base_info.update({
                'price_per_night': cheapest.get('eurPricePerNight'),
                'total_price': cheapest.get('totalPrice', {}).get('value'),
                'currency': cheapest.get('totalPrice', {}).get('currencyCode'),
                'bedroom_count': cheapest.get('bedRoomCount'),
                'provider': cheapest.get('providerName'),
                'provider_code': cheapest.get('providerCode'),
                'booking_url': cheapest.get('deeplinkUrl'),
                'room_type': cheapest.get('text'),
                'external_id': cheapest.get('externalId'),
                'from_date': cheapest.get('fromDate'),
                'to_date': cheapest.get('toDate')
            })
            
            # Thumbnail kép
            thumbnails = entry.get('lightThumbnails', {})
            first_urls = thumbnails.get('firstUrls', [])
            base_info['image_url'] = first_urls[0] if first_urls else None
        
        parsed_data.append(base_info)
    
    # DataFrame létrehozása
    df = pd.DataFrame(parsed_data)
    
    # Rendezés ár szerint
    if 'price_per_night' in df.columns:
        df = df.sort_values('price_per_night', ascending=True).reset_index(drop=True)
    
    print(f"✅ {len(df)} szállás feldolgozva!")
    
    return df


# Használati példa:
df = parse_cozycozy_results(results)
print(df[['name', 'price_per_night', 'rating_score', 'provider']].head(10))

✅ 90 szállás feldolgozva!
                             name  price_per_night  rating_score     provider
0              Mp Hostel Budapest        28.944000          82.0  hostelworld
1    Zen Hostel By Central Market        31.367999          87.0  hostelworld
2             Grand Richter Hotel        34.500000          75.0   hotels.com
3                   Avenue Hostel        38.128000          86.0  hostelworld
4                  Casa Del Hodos        39.239999          84.0  booking.com
5                 Mustache Hostel        41.516000          96.0  booking.com
6                 Wow City Hostel        42.579999          87.0  booking.com
7  The Hive Party Hostel Budapest        45.039999          88.0  hostelworld
8              Maverick Athenaeum        45.451999          87.0  hostelworld
9             Hermina Apartmanház        46.000000          80.0  booking.com


In [3]:
for col in df.columns:
    print(f"{col}: {df.loc[0, col]}")

accommodation_id: 12929817.0
name: Mp Hostel Budapest
title: Hostel
subtitle: Mp Hostel Budapest
city: Budapest
rating_score: 82.0
rating_count: 7427.0
location_text: 2 km from the city center
surface_text: 
latitude: 47.49691390991211
longitude: 19.066364288330078
cancellation_category: 0.0
instant_booking: True
result_count: 21.0
price_per_night: 28.944000244140625
total_price: 144.72000122070312
currency: EUR
bedroom_count: 1.0
provider: hostelworld
provider_code: hostelworld
booking_url: https://hostelworld.prf.hn/click/camref:1100l3xnx/pubref:%CLICK_ID%/[p_id:1011l373329]/destination:https%3A%2F%2Fwww.hostelworld.com%2F%2Fpwa%2Fhosteldetails.php%2Fhostel%2Fcity%2F306467%3Ffrom%3D2026-06-15%26to%3D2026-06-20%26guests%3D2
room_type: 2/12 beds in dorm
external_id: 306467
from_date: 2026-06-15
to_date: 2026-06-20
image_url: https://q-xx.bstatic.com/xdata/images/hotel/max600/369191423.jpg?k=538015023fe92cbe7e5ca6967818efb735ecb4350f7ab8282ea6b597323ee81b&o=


In [4]:
# Value
df.rating_score*(1-10**(-df.rating_count/50)) / df.price_per_night

0     2.833057
1     2.773527
2     2.173913
3     2.255560
4     2.140588
        ...   
85    1.250386
86    1.357828
87    1.249383
88         NaN
89         NaN
Length: 90, dtype: float64

In [5]:
# Városközpont koordináták

import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# km távolság kinyerése a location_text mezőből
df = df.copy()

df["distance_to_center_km"] = (
    df["location_text"]
    .str.extract(r"([\d\.]+)\s*km", expand=False)
    .astype(float)
)

# dobjuk ki azokat, ahol nincs értelmezhető távolság
df = df.dropna(subset=["latitude", "longitude", "distance_to_center_km"])

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Föld sugara km-ben

    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def residuals(center, latitudes, longitudes, distances):
    lat_c, lon_c = center
    predicted = haversine(latitudes, longitudes, lat_c, lon_c)
    return predicted - distances

latitudes = df["latitude"].values
longitudes = df["longitude"].values
distances = df["distance_to_center_km"].values

# kezdeti becslés: egyszerű átlag
initial_guess = [
    latitudes.mean(),
    longitudes.mean()
]

result = least_squares(
    residuals,
    x0=initial_guess,
    args=(latitudes, longitudes, distances),
    method="lm"  # Levenberg–Marquardt
)

city_center_lat, city_center_lon = result.x

city_center_lat, city_center_lon


(np.float64(47.498051304991975), np.float64(19.040033773738102))